# Module 7: Tối ưu không ràng buộc: điều kiện cực trị và phương pháp số — code

Notebook đi kèm site *Tối ưu hóa phi tuyến & lồi*. Chạy tuần tự từng ô (Colab có sẵn numpy, scipy, sympy, cvxpy). Tự giải tay trước rồi đối chiếu kết quả.

## Điểm dừng và phân loại bằng sympy (dạng A)

f = x⁴ + y⁴ − 4xy + 1: ba điểm dừng.

In [1]:
import sympy as sp, numpy as np
x, y = sp.symbols("x y", real=True); f = x**4 + y**4 - 4*x*y + 1
for s in sp.solve([sp.diff(f, x), sp.diff(f, y)], [x, y], dict=True):
    H = np.array(sp.hessian(f, (x, y)).subs(s).tolist(), dtype=float)
    print(s, "f =", f.subs(s), "giá trị riêng", np.linalg.eigvalsh(H).round(3))

{x: -1, y: -1} f = -1 giá trị riêng [ 8. 16.]
{x: 0, y: 0} f = 1 giá trị riêng [-4.  4.]
{x: 1, y: 1} f = -1 giá trị riêng [ 8. 16.]


## Gradient: tìm kiếm chính xác so với bước cố định (dạng D)

Ví dụ 2: 7 bước chính xác; t = 0.05 cần 68 bước.

In [2]:
import numpy as np
A = np.diag([2., 4]); b = np.array([-12., -12])
x = np.zeros(2); k = 0
while np.linalg.norm(A @ x + b) >= 1e-2:
    g = A @ x + b; x = x - (g @ g) / (g @ A @ g) * g; k += 1
print("chính xác:", k, x.round(4))
x = np.zeros(2); k = 0
while np.linalg.norm(A @ x + b) >= 1e-2: x = x - 0.05 * (A @ x + b); k += 1
print("cố định t=0.05:", k, x.round(4))

chính xác: 7 [5.9973 3.0014]
cố định t=0.05: 68 [5.9954 3.    ]


## Newton cho Bài tập 6 (dạng F)

3 bước, nghiệm (0.69588, −1.34794).

In [3]:
import numpy as np
g = lambda v: np.array([4*v[0]**3 + v[1], v[0] + 2*(1 + v[1])]); H = lambda v: np.array([[12*v[0]**2, 1], [1, 2]])
x = np.array([0.75, -1.25])
for k in range(5):
    print(k, x.round(10), round(float(np.linalg.norm(g(x))), 10))
    if np.linalg.norm(g(x)) < 1e-9: break
    x = x - np.linalg.solve(H(x), g(x))

0 [ 0.75 -1.25] 0.5038911093
1 [ 0.7  -1.35] 0.022
2 [ 0.69591078 -1.34795539] 0.0001401889
3 [ 0.69588439 -1.34794219] 5.8e-09
4 [ 0.69588439 -1.34794219] 0.0


## Gradient liên hợp: ≤ n bước (dạng H)

Ma trận Q ngẫu nhiên xác định dương.

In [4]:
import numpy as np
rng = np.random.default_rng(1)
for n in (2, 3, 4):
    M = rng.normal(size=(n, n)); Q = M @ M.T + n*np.eye(n); c = rng.normal(size=n)
    x = np.zeros(n); g = Q @ x + c; d = -g; k = 0
    while np.linalg.norm(g) > 1e-9:
        t = -(g @ d) / (d @ Q @ d); x = x + t*d; g2 = Q @ x + c; d = -g2 + (g2 @ g2)/(g @ g)*d; g = g2; k += 1
    print("n =", n, "số bước", k, "sai số", float(np.linalg.norm(x - np.linalg.solve(Q, -c))))

n = 2 số bước 2 sai số 0.0
n = 3 số bước 3 sai số 2.8028670759448284e-17
n = 4 số bước 4 sai số 1.962615573354719e-17


## Bài code · Đếm số bước gradient theo số điều kiện

Đầu vào: $\kappa$ (f = ½ x₁² + ½ κ x₂², bắt đầu tại (κ, 1) so với nghiệm). Đầu ra: số bước gradient chính xác đến ‖g‖ &lt; 1e-6.

In [5]:
import numpy as np
def steps_for(kappa):
    A = np.diag([1., kappa]); x = np.array([kappa, 1.]); k = 0
    while np.linalg.norm(A @ x) >= 1e-6:
        g = A @ x; x = x - (g @ g) / (g @ A @ g) * g; k += 1
    return k
print([steps_for(k) for k in (2, 10, 50)])

[14, 83, 452]


## Bài code · Phân loại điểm dừng bằng giá trị riêng

Đầu vào: ma trận Hessian đối xứng. Đầu ra: chuỗi “min”, “max”, “saddle” hoặc “?”.

In [6]:
import numpy as np
def kind(H, tol=1e-9):
    w = np.linalg.eigvalsh(np.array(H, float))
    if w.min() > tol: return "min"
    if w.max() < -tol: return "max"
    if w.min() < -tol and w.max() > tol: return "saddle"
    return "?"
print(kind([[12, -4], [-4, 12]]), kind([[0, -4], [-4, 0]]), kind([[0, 0], [0, 2]]))

min saddle ?
